In [32]:
import MySQLdb  

connection = MySQLdb.connect(
    host="localhost",  
    user="root",       
    password="db",     
    database="nowa"  
)

cursor = connection.cursor()

In [33]:
cursor.execute("SHOW TABLES;")
tables = cursor.fetchall()

print("Tabele w bazie danych:")
for table in tables:
    print(table[0])  

Tabele w bazie danych:
budzet
cele_oszczednosciowe
dochod
dochody_backup
dom
dom_backup
domownicy
domownicy_backup
kategorie
oszczednosci_domy
podsumowanie_domu
reklamanajczesciejwyswietlana
reklamy
widok_dochody
widok_grudzien_dochody
widok_grudzien_wydatki
widok_wydatki
wydatki
wydatki_backup
wyswietlone


In [34]:
# WYKRESY WYDATKÓW Z MIESIĄCA GRUDZIEŃ DLA KAŻDEGO UŻYTKOWNIKA

import MySQLdb
import pandas as pd
import plotly.express as px

connection = MySQLdb.connect(
    host="localhost",
    user="root",
    password="db",
    database="nowa",
    charset="utf8mb4"
)

query = """
SELECT nazwa_uzytkownika, kategoria_wydatku, wartosc_wydatku
FROM widok_grudzien_wydatki
"""
wydatki = pd.read_sql_query(query, connection)

def plot_interactive_pie_chart(data, user_column, category_column, value_column, title_prefix):
    users = data[user_column].unique()
    for user in users:
        user_data = data[data[user_column] == user]
        
        fig = px.pie(
            user_data,
            names=category_column,
            values=value_column,
            title=f"{title_prefix} dla użytkownika {user}",
            color_discrete_sequence=px.colors.qualitative.Pastel  
        )
        
        fig.show()

print("Wydatki z widoku grudzień:")
plot_interactive_pie_chart(wydatki, 'nazwa_uzytkownika', 'kategoria_wydatku', 'wartosc_wydatku', "Wydatki")

Wydatki z widoku grudzień:


C:\Users\jglod\AppData\Local\Temp\ipykernel_18844\322402217.py:19: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



In [35]:
# WYKRES OSZCZĘDNOŚCI DOMÓW Z KAŻDEGO MIESIĄCA ROKU 2024

import MySQLdb
import pandas as pd
import plotly.express as px

connection = MySQLdb.connect(
    host="localhost",  
    user="root",       
    password="db",     
    database="nowa"    
)

cursor = connection.cursor()

query_oszczednosci = """
SELECT 
    nazwa_domu, 
    miesiac, 
    oszczednosci
FROM 
    oszczednosci_domy
ORDER BY 
    nazwa_domu, miesiac;
"""

df_oszczednosci = pd.read_sql(query_oszczednosci, connection)

fig = px.line(
    df_oszczednosci,
    x='miesiac',
    y='oszczednosci',
    color='nazwa_domu',  
    markers=True,        
    title="Oszczędności Domów w 2024 roku",
    labels={"miesiac": "Miesiąc", "oszczednosci": "Oszczędności (Dochody - Wydatki)"},
)

fig.update_layout(
    xaxis=dict(tickmode='array', tickvals=list(range(1, 13))),  
    xaxis_title="Miesiąc",
    yaxis_title="Oszczędności (Dochody - Wydatki)",
    legend_title="Domy",
    template="plotly_white",  
)

fig.show()

C:\Users\jglod\AppData\Local\Temp\ipykernel_18844\2332766992.py:27: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



In [36]:
# WYKRES ZAOSZCZĘDZONYCH PIENIĘDZY W 2024 ROKU DLA KAŻDEGO UŻYTKOWNIKA 

import MySQLdb
import pandas as pd
import plotly.express as px

connection = MySQLdb.connect(
    host="localhost",  
    user="root",       
    password="db",     
    database="nowa"    
)

cursor = connection.cursor()

query = """
SELECT nazwa_uzytkownika, zaoszczedzona_kwota
FROM budzet
"""
data = pd.read_sql_query(query, connection)

cursor.close()
connection.close()

print(data)

fig = px.bar(
    data,
    x='nazwa_uzytkownika',            
    y='zaoszczedzona_kwota',           
    title="Zaoszczędzone kwoty przez użytkowników",
    labels={"nazwa_uzytkownika": "Użytkownik", "zaoszczedzona_kwota": "Zaoszczędzona kwota (zł)"},
    color='nazwa_uzytkownika',         
    color_discrete_sequence=px.colors.qualitative.Set2  
)

fig.update_layout(
    xaxis_title="Użytkownik",
    yaxis_title="Zaoszczędzona kwota (zł)",
    xaxis_tickangle=-45,   
    template="plotly_white"  
)

fig.show()

C:\Users\jglod\AppData\Local\Temp\ipykernel_18844\1477098383.py:20: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



           nazwa_uzytkownika  zaoszczedzona_kwota
0          adam.jabłoński667                21638
1           ewa.jabłońska270                23874
2         kamil.jabłoński150                 7760
3         anna.wiśniewska739                21089
4        piotr.wiśniewski448                26867
5        julia.wiśniewska822                20235
6       oliwia.wiśniewska867                17830
7      mateusz.wiśniewski870                 2724
8           jan.dąbrowski750                26867
9         maria.dąbrowska139                43752
10       tomasz.dąbrowski146                32159
11      natalia.dąbrowska213                16694
12      artur.lewandowski528                25905
13  magdalena.lewandowska100                35995
14          karol.nowicki619                26032
15         joanna.nowicka896                20212
16          zofia.nowicka724                10218
17         michał.nowicki831                24785
18            paweł.mazur982                33847


In [37]:
# WYKRES POSTĘPÓW W REALIZACJI CELÓW DLA KAŻDEGO UŻYTKOWNIKA

import MySQLdb
import pandas as pd
import plotly.graph_objects as go

connection = MySQLdb.connect(
    host="localhost",  
    user="root",       
    password="db",     
    database="nowa"    
)

cursor = connection.cursor()

query = """
SELECT nazwa_uzytkownika, docelowa_kwota, zaoszczedzona_kwota, brakujaca_kwota
FROM cele_oszczednosciowe
"""
cursor.execute(query)
data = cursor.fetchall()

df = pd.DataFrame(data, columns=["nazwa_uzytkownika", "docelowa_kwota", "zaoszczedzona_kwota", "brakujaca_kwota"])

fig = go.Figure()

fig.add_trace(go.Bar(
    x=df["nazwa_uzytkownika"],
    y=df["zaoszczedzona_kwota"],
    name="Zaoszczędzona Kwota",
    marker_color='blue'
))

fig.add_trace(go.Bar(
    x=df["nazwa_uzytkownika"],
    y=df["brakujaca_kwota"],
    name="Brakująca Kwota",
    marker_color='orange'
))

fig.add_trace(go.Bar(
    x=df["nazwa_uzytkownika"],
    y=df["docelowa_kwota"],
    name="Docelowa Kwota",
    marker_color='green'
))

fig.update_layout(
    title="Porównanie celów oszczędnościowych",
    xaxis_title="Użytkownicy",
    yaxis_title="Kwota (PLN)",
    barmode='group',  
    xaxis_tickangle=-45,  
    template="plotly_white",  
    legend_title="Typ Kwoty"
)

fig.show()

In [38]:
# WYKRES PROCENTOWY POSTĘPÓW W REALIZACJI CELÓW DLA KAŻDEGO UŻYTKOWNIKA

import MySQLdb
import pandas as pd
import plotly.graph_objects as go

# Połączenie z bazą danych
connection = MySQLdb.connect(
    host="localhost",
    user="root",
    password="db",
    database="nowa",
    charset="utf8mb4"
)

# Zapytanie SQL do pobrania danych z bazy
query = """
SELECT
    nazwa_uzytkownika,
    opis_celu,
    docelowa_kwota,
    COALESCE(zaoszczedzona_kwota, 0) AS zaoszczedzona_kwota
FROM cele_oszczednosciowe
WHERE docelowa_kwota IS NOT NULL;
"""

# Wczytanie danych do pandas
cele = pd.read_sql_query(query, connection)

# Obliczenie postępu realizacji celu
cele['postep'] = (cele['zaoszczedzona_kwota'] / cele['docelowa_kwota']) * 100

# Tworzenie wykresu
fig = go.Figure()

# Dodanie wykresu słupkowego z etykietami
fig.add_trace(go.Bar(
    y=cele['nazwa_uzytkownika'] + " - " + cele['opis_celu'],
    x=cele['postep'],
    orientation='h',
    marker_color='green',
    name="Postęp realizacji celu"
))

# Zaktualizowanie układu wykresu
fig.update_layout(
    title="Postęp w realizacji celów oszczędnościowych",
    xaxis_title="Procent postępu (%)",
    yaxis_title="Użytkownik i Cel",
    template="plotly_white",
    margin={"l": 250},  # Większy margines, aby etykiety były czytelne
    showlegend=False
)

# Wyświetlenie wykresu
fig.show()


C:\Users\jglod\AppData\Local\Temp\ipykernel_18844\3104339275.py:28: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



In [39]:
# WYKRES ZAOSZCZĘDZONEJ KWOTY W POSZCZEGÓLNYCH DOMACH 

import MySQLdb
import pandas as pd
import plotly.graph_objects as go

connection = MySQLdb.connect(
    host="localhost", 
    user="root",       
    password="db",     
    database="nowa"    
)

cursor = connection.cursor()

query = """
SELECT nazwa_domu, zaoszczedzona_kwota
FROM podsumowanie_domu;
"""
cursor.execute(query)

data = cursor.fetchall()  
df = pd.DataFrame(data, columns=["nazwa_domu", "zaoszczedzona_kwota"])

fig = go.Figure()

fig.add_trace(go.Bar(
    x=df['nazwa_domu'], 
    y=df['zaoszczedzona_kwota'],  
    marker_color='skyblue', 
))

fig.update_layout(
    title="Oszczędności w poszczególnych domach",  
    xaxis_title="Domy",  
    yaxis_title="Zaoszczędzona Kwota (PLN)", 
    xaxis_tickangle=45,  
    template="plotly_white",  
)

fig.show()

In [40]:
# WYKRES PORÓWNUJACY MIESIĘCZNY DOCHÓD W KAŻDYM DOMU

query = """
SELECT
    d.nazwa_domu,
    MONTH(dochod.data_dochodu) AS miesiac,
    SUM(dochod.wartosc_dochodu) AS dochod
FROM dochod
JOIN domownicy d ON dochod.nazwa_uzytkownika = d.nazwa_uzytkownika
WHERE YEAR(dochod.data_dochodu) = 2024
GROUP BY d.nazwa_domu, MONTH(dochod.data_dochodu)
ORDER BY d.nazwa_domu, miesiac;
"""

df = pd.read_sql(query, connection)

print(df.head())

fig = px.bar(df, x='miesiac', y='dochod', color='nazwa_domu',
             labels={'miesiac': 'Miesiąc', 'dochod': 'Dochód'},
             title='Dochody domów w 2024 roku')

fig.show()

C:\Users\jglod\AppData\Local\Temp\ipykernel_18844\2530588823.py:15: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



  nazwa_domu  miesiac   dochod
0    Artyści        1  21339.0
1    Artyści        2  26345.0
2    Artyści        3  26424.0
3    Artyści        4  23923.0
4    Artyści        5  24106.0


In [41]:
# WYKRES ZESTAWIENIA WYDATKÓW DOMÓW W KAŻDYM MIESIĄCU

import MySQLdb
import pandas as pd
import plotly.graph_objects as go

connection = MySQLdb.connect(
    host="localhost",  
    user="root",       
    password="db",     
    database="nowa"    
)

cursor = connection.cursor()
query = """
SELECT
    d.nazwa_domu,
    MONTH(w.data_wydatku) AS miesiac,
    w.kategoria_wydatku,
    SUM(w.wartosc_wydatku) AS wartosc_wydatku
FROM wydatki w
JOIN domownicy d ON w.nazwa_uzytkownika = d.nazwa_uzytkownika
WHERE YEAR(w.data_wydatku) = 2024
GROUP BY d.nazwa_domu, MONTH(w.data_wydatku), w.kategoria_wydatku
ORDER BY d.nazwa_domu, miesiac, w.kategoria_wydatku;
"""

df = pd.read_sql(query, connection)

connection.close()

df.head()

df_grouped = df.groupby(['nazwa_domu', 'miesiac', 'kategoria_wydatku'], as_index=False).agg({'wartosc_wydatku': 'sum'})

fig = px.bar(df_grouped, 
             x='miesiac', 
             y='wartosc_wydatku', 
             color='kategoria_wydatku', 
             facet_col='nazwa_domu',  
             labels={'miesiac': 'Miesiąc', 'wartosc_wydatku': 'Wartość Wydatku (PLN)'},
             title="Wydatki w skali roku z podziałem na kategorie")

fig.show()

C:\Users\jglod\AppData\Local\Temp\ipykernel_18844\3571414753.py:28: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



In [52]:
# WYKRES PORÓWNANIA ZSUMOWANYCH DOCHODÓW I WYDATKÓW WSZYSTKICH UŻYTKOWNIKÓW W CIĄGU ROKU 

import MySQLdb
import pandas as pd
import plotly.graph_objects as go

connection = MySQLdb.connect(
    host="localhost",  
    user="root",       
    password="db",     
    database="nowa"    
)

query = """
SELECT 
    MONTH(d.data_dochodu) AS miesiac,
    SUM(d.wartosc_dochodu) AS dochody,
    (SELECT SUM(w.wartosc_wydatku) 
     FROM wydatki w 
     WHERE MONTH(w.data_wydatku) = MONTH(d.data_dochodu) 
     AND YEAR(w.data_wydatku) = 2024) AS wydatki
FROM dochod d
WHERE YEAR(d.data_dochodu) = 2024
GROUP BY miesiac
ORDER BY miesiac;
"""

df = pd.read_sql(query, connection)

fig = px.bar(df, 
             x='miesiac', 
             y=['dochody', 'wydatki'], 
             title='Porównanie dochodów i wydatków w ciągu roku 2024',
             labels={'miesiac': 'Miesiąc', 'value': 'Kwota', 'variable': 'Kategorie'},
             barmode='group',  
             color_discrete_sequence=['green', 'red'])  

fig.show()

connection.close()

C:\Users\jglod\AppData\Local\Temp\ipykernel_18844\3671897918.py:28: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



In [53]:
# WYKRES PORÓWNANIA ROCZNYCH ZSUMOWANYCH DOCHODÓW I WYDATKÓW DOMÓW 

import MySQLdb
import pandas as pd
import plotly.graph_objects as go

connection = MySQLdb.connect(
    host="localhost",  
    user="root",      
    password="db",     
    database="nowa"  
)

query = """
SELECT 
    d.nazwa_domu,
    SUM(dt.wartosc_dochodu) AS dochody,
    (SELECT SUM(w.wartosc_wydatku) 
     FROM wydatki w 
     WHERE w.nazwa_domu = d.nazwa_domu 
     AND YEAR(w.data_wydatku) = 2024) AS wydatki
FROM dom d
LEFT JOIN dochod dt ON d.nazwa_domu = dt.nazwa_domu
WHERE YEAR(dt.data_dochodu) = 2024
GROUP BY d.nazwa_domu;
"""

df = pd.read_sql(query, connection)

fig = px.bar(df, 
             x='nazwa_domu', 
             y=['dochody', 'wydatki'], 
             title='Porównanie dochodów i wydatków w domach w 2024 roku',
             labels={'nazwa_domu': 'Nazwa domu', 'value': 'Kwota', 'variable': 'Kategorie'},
             barmode='group',  
             color_discrete_sequence=['green', 'red']) 

fig.show()

connection.close()

C:\Users\jglod\AppData\Local\Temp\ipykernel_18844\1285868022.py:28: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



In [ ]:
# WYKRES PROCENTOWEGO UDZIAŁU LICZBY WYŚWIETLEŃ REKLAM

import pymysql
import pandas as pd
import plotly.express as px

connection = pymysql.connect(
    host="localhost",  
    user="root",       
    password="db",     
    database="nowa",   
    charset="utf8mb4"  
)

query = "SELECT nazwa_reklamy, liczba_wyswietlen FROM ReklamaNajczesciejWyswietlana"
df = pd.read_sql(query, connection)

fig = px.pie(
    df, 
    names='nazwa_reklamy', 
    values='liczba_wyswietlen', 
    title='Procentowy udział liczby wyświetleń reklam',
    color_discrete_sequence=px.colors.qualitative.Pastel  
)

fig.update_traces(hole=0.2)  

fig.show()

connection.close()

C:\Users\jglod\AppData\Local\Temp\ipykernel_18844\1140263066.py:14: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



In [50]:
# WYKRES ZALEŻNOŚCI KLIKNIĘĆ REKLAM

import pymysql
import pandas as pd
import plotly.express as px

connection = pymysql.connect(
    host="localhost",  
    user="root",       
    password="db",     
    database="nowa",   
    charset="utf8mb4"  
)

query = """
SELECT r.nazwa_reklamy, 
       COUNT(w.id_reklamy) AS liczba_klikniec
FROM reklamy r
JOIN wyswietlone w ON r.id = w.id_reklamy
WHERE w.klikniete = TRUE
GROUP BY r.nazwa_reklamy
ORDER BY liczba_klikniec DESC
"""

df = pd.read_sql(query, connection)

fig = px.bar(
    df, 
    x='nazwa_reklamy', 
    y='liczba_klikniec', 
    title='Najczęściej kliknięte reklamy',
    color='liczba_klikniec',
    color_continuous_scale='Blues'  
)

fig.show()

connection.close()

C:\Users\jglod\AppData\Local\Temp\ipykernel_18844\2054983442.py:23: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



In [55]:
# WYKRES PROCENTOWEGO UDZIAŁU LICZBY KLIKNIĘĆ REKLAM

import pymysql
import pandas as pd
import plotly.express as px

# Połączenie z bazą danych
connection = pymysql.connect(
    host="localhost",  
    user="root",       
    password="db",     
    database="nowa",   
    charset="utf8mb4"  
)

# Zapytanie SQL dla najczęściej klikniętych reklam
query = "SELECT nazwa_reklamy, liczba_klikniec FROM ReklamaNajczesciejKliknieta"
df = pd.read_sql(query, connection)

# Wykres kołowy dla najczęściej klikniętych reklam
fig = px.pie(
    df, 
    names='nazwa_reklamy', 
    values='liczba_klikniec', 
    title='Procentowy udział liczby kliknięć reklam',
    color_discrete_sequence=px.colors.qualitative.Pastel  
)

# Ustawienie dziury w środku wykresu (aby był podobny do wykresu pierścieniowego)
fig.update_traces(hole=0.2)  

# Wyświetlenie wykresu
fig.show()

# Zamknięcie połączenia z bazą danych
connection.close()

C:\Users\jglod\AppData\Local\Temp\ipykernel_18844\2992428480.py:16: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



APLIKACJA BUDŻETOWNIK

In [56]:
import MySQLdb
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import tkinter as tk
from tkinter import ttk

connection = MySQLdb.connect(
    host="localhost",
    user="root",
    password="db",
    database="nowa",
    charset="utf8mb4"
)

def get_user_data(username):
    query_expenses = """
    SELECT kategoria_wydatku, SUM(wartosc_wydatku) AS total
    FROM widok_grudzien_wydatki
    WHERE nazwa_uzytkownika = %s
    GROUP BY kategoria_wydatku
    """
    query_income = """
    SELECT kategoria_dochodu, SUM(wartosc_dochodu) AS total
    FROM widok_grudzien_dochody
    WHERE nazwa_uzytkownika = %s
    GROUP BY kategoria_dochodu
    """
    expenses_data = pd.read_sql_query(query_expenses, connection, params=(username,))
    income_data = pd.read_sql_query(query_income, connection, params=(username,))
    return expenses_data, income_data

def plot_user_charts(expenses_data, income_data, user):
    if expenses_data.empty and income_data.empty:
        show_error_message("Brak danych dla podanego użytkownika.")
        return
    
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Wydatki", "Dochody"),
        specs=[[{"type": "pie"}, {"type": "pie"}]]
    )
    fig.add_trace(
        go.Pie(
            labels=expenses_data['kategoria_wydatku'],
            values=expenses_data['total'],
            name=f"Wydatki dla użytkownika {user}",
            marker=dict(colors=px.colors.sequential.Plasma)
        ),
        row=1, col=1
    )
    fig.add_trace(
        go.Pie(
            labels=income_data['kategoria_dochodu'],
            values=income_data['total'],
            name=f"Dochody dla użytkownika {user}",
            marker=dict(colors=px.colors.sequential.Turbo)
        ),
        row=1, col=2
    )
    fig.update_layout(
        title_text=f"Wydatki i dochody dla użytkownika {user}",
        showlegend=True,
        plot_bgcolor='rgb(242, 242, 242)',
        title_font=dict(size=20, family="Arial, sans-serif", color="black"),
    )
    fig.show()

def show_success_message(user):
    success_window = tk.Toplevel(app)
    success_window.title("Sukces")
    success_window.geometry("300x150")
    success_window.configure(bg="#90EE90")

    message_label = ttk.Label(success_window, text=f"Zalogowano jako:\n{user}", 
                               font=("Arial", 12), background="#90EE90", foreground="black")
    message_label.pack(pady=30)

    close_button = ttk.Button(success_window, text="Zamknij", command=success_window.destroy, style="TButton")
    close_button.pack()

def show_error_message(message="Niepoprawna nazwa użytkownika lub hasło."):
    error_window = tk.Toplevel(app)
    error_window.title("Błąd logowania")
    error_window.geometry("300x150")
    error_window.configure(bg="#FF6666")

    message_label = ttk.Label(error_window, text=message, 
                               font=("Arial", 12), background="#FF6666", foreground="black")
    message_label.pack(pady=30)

    close_button = ttk.Button(error_window, text="Zamknij", command=error_window.destroy, style="TButton")
    close_button.pack()

def login():
    username = username_entry.get()
    password = password_entry.get()
    
    query = """
    SELECT nazwa_uzytkownika
    FROM domownicy
    WHERE nazwa_uzytkownika = %s AND haslo = %s
    """
    cursor = connection.cursor()
    cursor.execute(query, (username, password))
    result = cursor.fetchone()
    
    if result:
        show_success_message(username)
        expenses_data, income_data = get_user_data(username)
        plot_user_charts(expenses_data, income_data, username)
    else:
        show_error_message()

In [57]:
app = tk.Tk()
app.title("Logowanie do aplikacji")
app.geometry("400x300")
app.configure(bg="#ADD8E6")  

style = ttk.Style()
style.configure('TButton', font=('Arial', 12), padding=10)
style.configure('TLabel', font=('Arial', 12))
style.configure('TEntry', font=('Arial', 12))

header_label = tk.Label(app, text="Witamy w Budżetownik", bg="#ADD8E6", fg="#333333", 
                        font=("Arial", 16, "bold"))
header_label.pack(pady=(20, 5))  

subheader_label = tk.Label(app, text="Zaloguj się do aplikacji", bg="#ADD8E6", fg="#666666", 
                           font=("Arial", 12))
subheader_label.pack(pady=(10, 10))  

frame = tk.Frame(app, bg="#ADD8E6")
frame.pack(pady=10)

username_label = ttk.Label(frame, text="Nazwa użytkownika:", background="#ADD8E6")
username_label.grid(row=0, column=0, padx=10, pady=10, sticky='w')

username_entry = ttk.Entry(frame, width=25)
username_entry.grid(row=0, column=1, padx=10, pady=10)

password_label = ttk.Label(frame, text="Hasło:", background="#ADD8E6")
password_label.grid(row=1, column=0, padx=10, pady=10, sticky='w')

password_entry = ttk.Entry(frame, width=25, show="*")
password_entry.grid(row=1, column=1, padx=10, pady=10)

login_button = ttk.Button(app, text="Zaloguj się", command=login)
login_button.pack(pady=20)

app.mainloop()

C:\Users\jglod\AppData\Local\Temp\ipykernel_18844\4012047678.py:30: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.

C:\Users\jglod\AppData\Local\Temp\ipykernel_18844\4012047678.py:31: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.

